# От фотографий iNaturalist до сравнения четырёх моделей

**Run All** запускает этапы `01 → 02 → 03a → 03b → 03c → 03d → модели` в одном ядре. Обычный запуск повторяет эксперимент команды. Исходные наблюдения — снимок выдачи iNaturalist, ручная разметка и ответы исполнителей. Отбор 400 + 100, загрузка оригиналов, построение эталона, инференс и оценка выполняются заново. Каждая ячейка ниже использует файлы или переменные этого прогона.

Для полного запуска нужны интернет, NVIDIA GPU с CUDA, зависимости инференса и собранный `llama-server`. Вычисления и датасеты пишутся в отдельные каталоги. Если нужна только офлайн-проверка опубликованных результатов, у неё есть отдельный ноутбук. [Подготовка окружения](docs/reproduce.md) · [Проверка сохранённых результатов](notebooks/reproduce_saved_results.ipynb)

## Режим запуска

- **historical** повторяет исходный эксперимент с теми же 700 кандидатами и ответами исполнителей. Оригиналы фотографий скачиваются, а четыре модели запускаются заново.
- **live** получает сегодняшнюю выдачу iNaturalist. Вы размечаете вспомогательные кадры в Jupyter, затем ноутбук открывает **платные пулы** в Яндекс Заданиях и ждёт ответов. Нужен токен заказчика `YANDEX_TASKS_OAUTH_TOKEN`.

Каждый **Run All** начинает новый прогон. Время запуска используется в именах папок с данными и результатами.

In [ ]:
MODE = "historical"  # historical | live

In [ ]:
import json
import os
import sys
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path
from tempfile import NamedTemporaryFile

from IPython.display import Markdown, display
from PIL import Image

root = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "pyproject.toml").is_file() and (p / "butterfly_bench").is_dir())
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
from butterfly_bench.schema import validate_boxes
from scripts.download_selected import checksum, jsonl_rows



if MODE not in ("historical", "live"):
    raise ValueError("Выберите historical или live")

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
run_dir = root / "outputs/pipeline" / stamp
run_dir.mkdir(parents=True)
downloads = root.parent / "downloads/final-project"
image_dir = downloads / "pipeline" / stamp / "images"
os.environ.setdefault("HF_HOME", str(downloads / "huggingface"))
print("Результаты:", run_dir, "\nОригиналы:", image_dir)

In [ ]:
def indexed(rows):
    by_id = {row["photo_id"]: row for row in rows}
    if len(by_id) != len(rows):
        raise ValueError("Повторяется photo_id")
    return by_id

def write_rows(path, rows):
    path.parent.mkdir(parents=True, exist_ok=True)
    with NamedTemporaryFile("w", dir=path.parent, encoding="utf-8", delete=False) as stream:
        for row in rows:
            stream.write(json.dumps(row, ensure_ascii=False, allow_nan=False) + "\n")
    Path(stream.name).replace(path)

def table(headers, rows):
    lines = [headers, ["---"] * len(headers), *rows]
    display(Markdown("\n".join("| " + " | ".join(map(str, row)) + " |" for row in lines)))

## 01. Формирование выборки iNaturalist

На первом шаге собираются 700 кандидатов из четырёх категорий наблюдений. Из каждой категории выбираем снимки для основной и вспомогательной выборок по квотам первоначального эксперимента. Конкурсные фотографии исключаются до отбора. [Исходный ноутбук 01](notebooks/data/01_build_photo_manifests.ipynb) · [План отбора](data/sampling/sampling_plan.json)

In [ ]:
plan = json.loads((root / "data/sampling/sampling_plan.json").read_text())
archive = root / "data/sampling/runs/20261108T001420_435829Z"
model_archive = root / "results/final_400"
frozen_plan = json.loads((model_archive / "plan.json").read_text())
assert plan["seed"] == 20261107 and frozen_plan["seed"] == 42
metadata = {"mode": MODE, "sampling_seed": plan["seed"], "model_seed": frozen_plan["seed"]}
(run_dir / "run.json").write_text(json.dumps(metadata, indent=2) + "\n")

def photo_urls(rows):
    return {i: row["photo_url"] for i, row in indexed(rows).items()}

contest = jsonl_rows(root / "data/contest_400/photos.jsonl")
template = jsonl_rows(root / "data/contest_400/submission-template.jsonl")
assert len(contest) == 400 and photo_urls(contest) == photo_urls(template)

groups = plan["groups"]
table(["Категория", "Кандидаты", "Основные", "Вспомогательные"],
      [[g["name"], g["candidate_count"], g["target_count"], g["training_exam_count"]] for g in groups])
print("Seed:", plan["seed"])

### 1. Получение кандидатов через API

В режиме `historical` читаем весь снимок выдачи iNaturalist от 8 ноября 2026 года. В режиме `live` запрашиваем страницы API по тем же таксонам и квотам. Сохраняем кандидатов до случайного отбора, чтобы было видно, из чего получились итоговые выборки.

In [ ]:
from scripts.pipeline_sampling import collect_candidates

snapshot = archive / "candidates_from_api.jsonl" if MODE == "historical" else None
candidates = collect_candidates(run_dir, plan, contest, snapshot=snapshot)
counts = Counter(row["source_stratum"] for row in candidates)
assert counts == {g["name"]: g["candidate_count"] for g in groups}
assert all(row["license_code"] == "cc0" for row in candidates)
print(len(candidates), "кандидатов CC0 сохранено в", run_dir / "candidates_from_api.jsonl")

### 2. Случайный отбор 400 + 100 фотографий

`suggest` выбирает без возвращения снимки с короткой стороной от 400 пикселей. Для каждой группы используется `seed + 1000 + номер группы`. Остальные кандидаты остаются в резерве. В CSV сохраняется назначение каждого из 700 фото.

In [ ]:
from scripts.prepare_review import suggest
from scripts.pipeline_sampling import select_photos

chosen = suggest(candidates, plan)
selected = select_photos(run_dir, candidates, plan, suggested=chosen)
main_selected = [r for r in selected if r["purpose"] == "main"]
ref_selected = [r for r in selected if r["purpose"] == "worker_train_exam"]
assert len(main_selected) == 400 and len(ref_selected) == 100
assert set(photo_urls(main_selected)).isdisjoint(photo_urls(ref_selected))
assert set(photo_urls(selected)).isdisjoint(photo_urls(contest))
if MODE == "historical":
    assert photo_urls(main_selected) == photo_urls(jsonl_rows(root / "data/golden_400/photos.jsonl"))
    assert photo_urls(ref_selected) == photo_urls(jsonl_rows(root / "data/reference_100/photos.jsonl"))
actual = Counter((r["source_stratum"], r["purpose"]) for r in selected)
table(["Категория", "Основные", "Вспомогательные"],
      [[g["name"], actual[g["name"], "main"], actual[g["name"], "worker_train_exam"]] for g in groups])
print("Назначения:", run_dir / "selection_500_proposal.csv")
print("Галерея кандидатов:", run_dir / "candidate_gallery_500.html")

### 3. Загрузка и проверка оригиналов

Скачиваем оригиналы, проверяем формат и размеры, считаем SHA-256 и ищем побайтовые дубликаты. Для исторического запуска каждый хеш сверяется с первоначальным журналом. Новый Run All сохраняет свои оригиналы отдельно от предыдущих запусков.

In [ ]:
from scripts.pipeline_sampling import download_originals

expected_hashes = None
if MODE == "historical":
    expected_hashes = {r["photo_id"]: r["sha256"] for r in jsonl_rows(archive / "download_audit.jsonl")}
audit = download_originals(run_dir, selected, image_dir, expected_hashes=expected_hashes)
print(f"Проверено {len(audit)} оригиналов, {sum(r['bytes'] for r in audit):,} байт")
print("Журнал проверки:", run_dir / "download_audit.jsonl")

### 4. Поиск визуально похожих фотографий

Вычисляем dHash каждого оригинала и сравниваем пары. В таблицу для визуальной проверки попадают пары с расстоянием Хэмминга не выше 4.

In [ ]:
from scripts.pipeline_sampling import review_similar_photos

pair_count = review_similar_photos(run_dir, audit)
print("Пар для визуальной проверки:", pair_count)
print(run_dir / "similarity_review.csv")

### 5. Сохранение манифестов и результатов отбора

Для каждого набора записываем ID, публичный URL и путь к скачанному файлу. Следующие шаги читают из этих манифестов свою выборку.

In [ ]:
from scripts.pipeline_sampling import save_photo_manifests

report = save_photo_manifests(run_dir, audit)
main_path = run_dir / "main/photos.jsonl"
reference_path = run_dir / "reference/photos.jsonl"
main = jsonl_rows(main_path)
reference_photos = jsonl_rows(reference_path)
assert len(main) == report["main_count"] == 400
assert len(reference_photos) == report["reference_count"] == 100
print(main_path, reference_path, run_dir / "sampling_audit.json", sep="\n")

## 02. Проверить разметку вспомогательной сотни

В исходном проекте эти сто фотографий разметили вручную. Для повтора читаем первичный JSON-массив разметчика, проверяем его связь с только что выбранными фотографиями и сохраняем проверенные рамки. Для новых кадров Jupyter показывает фотографии и принимает координаты. Ответ `none` означает, что человек осмотрел снимок и взрослых бабочек нет. [Исходный ноутбук 02](notebooks/data/02_review_worker_reference.ipynb)

In [ ]:
from scripts.pipeline_sampling import prepare_reference, review_reference, load_reference

if MODE == "historical":
    expert = json.loads((root / "data/reference_100/worker_reference_first_pass.json").read_text())
    if photo_urls(expert) != photo_urls(reference_photos):
        raise ValueError("Фотографии разметчика и выбранная сотня различаются")
    write_rows(run_dir / "reference/labels.jsonl", expert)
else:
    prepare_reference(run_dir)
    review_reference(run_dir)
reference = load_reference(run_dir)

### Проверка рамок и примеры

In [ ]:
reference_by_id = indexed(reference)
assert len(reference) == 100
assert photo_urls(reference) == photo_urls(reference_photos)
assert set(reference_by_id).isdisjoint(photo_urls(main))
assert set(reference_by_id).isdisjoint(photo_urls(contest))
for row in reference:
    validate_boxes(row["boxes"])
positive = sum(bool(row["boxes"]) for row in reference)
print("С бабочками:", positive, "| без бабочек:", len(reference) - positive,
      "| рамок:", sum(len(row["boxes"]) for row in reference))

In [ ]:
from matplotlib import pyplot as plt
from butterfly_bench.visualize import draw_prediction

def show_examples(start=0, count=12):
    photos = reference[start:start+count]
    if not photos:
        raise ValueError("В выбранном диапазоне нет фотографий")
    paths = indexed(reference_photos)
    fig, axes = plt.subplots((len(photos)+3)//4, 4, squeeze=False,
                             figsize=(12, 2.8 * ((len(photos)+3)//4)), layout="constrained")
    for axis in axes.flat:
        axis.axis("off")
    for axis, row in zip(axes.flat, photos):
        pid = row["photo_id"]
        with Image.open(reference_path.parent / paths[pid]["image_path"]) as image:
            frame = draw_prediction(image.convert("RGB"), {"boxes": row["boxes"], "error": None})
        frame.thumbnail((340, 260))
        axis.imshow(frame)
        axis.set_title(f"{pid} · рамок {len(row['boxes'])}", fontsize=9)
    plt.show()

show_examples()

## 03a. Экзамен разметчиков и первичный ответ

Исторический режим восстанавливает экзаменационный ключ и рабочие страницы для выбранных ID. Разметка людей сохранена во входах для независимых ревьюеров. Оттуда получаем статус, рамки и идентификатор выдачи каждого первого ответа.

В live создаются проект, навыки и экзамен. Рабочий пул получает выбранные 400 фото, по четыре на страницу. Обработка продолжится после ответов людей. [Исходный ноутбук 03a](notebooks/crowd/03a_create_annotation.ipynb)

In [ ]:
inputs = root / "crowd"
crowd_dir = run_dir / "crowd"
crowd_dir.mkdir(exist_ok=True)
main_urls = photo_urls(main)

def historical_answers(filename, expected_ids):
    pages = json.loads((inputs / "review/tasks" / filename).read_text())
    rows, controls = [], 0
    for page in pages:
        for task in page["tasks"]:
            if "known_solutions" in task:
                controls += 1
                continue
            v = task["input_values"]
            if v["image"] != main_urls[v["photo_id"]]:
                raise ValueError(f"URL разметки отличается для {v['photo_id']}")
            rows.append({"photo_id": v["photo_id"], "photo_url": v["image"], "status": v["worker_status"],
                         "source_assignment_id": v["source_assignment_id"], "boxes": validate_boxes(json.loads(v["boxes_json"]))})
    records = indexed(rows)
    if records.keys() != set(expected_ids):
        raise ValueError(f"Неполный набор ответов в {filename}")
    print(f"{filename}: {len(pages)} страниц, {len(rows)} рабочих заданий, {controls} контрольных")
    return records

### Экзамен и допуск к разметке

In [ ]:
if MODE == "historical":
    exam = jsonl_rows(inputs / "annotation/exam_key.jsonl")
    for task in exam:
        assert task["boxes"] == reference_by_id[task["photo_id"]]["boxes"]
    write_rows(crowd_dir / "annotation_exam_key.jsonl", exam)
    print("Фотографий в экзамене разметчиков:", len(exam))
else:
    from dotenv import load_dotenv
    from scripts.pipeline_crowd import CrowdRun
    load_dotenv(root / ".env")
    crowd = CrowdRun(run_dir, main, reference)
    qualification = crowd.qualify_annotators()
    print("Проект:", qualification["project_id"], "| пул экзамена:", qualification["exam_pool_id"])
    print("Прошли экзамен:", qualification["passed"], "из", qualification["participants"])

### Рабочая разметка 400 фотографий

In [ ]:
if MODE == "historical":
    work_pages = json.loads((inputs / "annotation/tasks/annotation_work.json").read_text())
    tasks = [t["input_values"] for p in work_pages for t in p["tasks"]]
    assert len(tasks) == len(main) and {t["photo_id"]: t["image"] for t in tasks} == main_urls
    print("Страниц в рабочем пуле:", len(work_pages))
    first = historical_answers("validation_work.json", main_urls)
else:
    first = crowd.annotate(qualification=qualification)
write_rows(crowd_dir / "first_answers.jsonl", first.values())
print("Статусы первой разметки:", dict(Counter(r["status"] for r in first.values())))

## 03b. Независимая проверка

При новом сборе исполнители проходят обучение и экзамен, затем три ревьюера проверяют каждый рабочий ответ. В историческом наборе сохранились итоговые количества голосов, список отправленных на переразметку и задания проверяющих. Для первоначального раунда архив содержит ограниченную информацию: набор 43 сформировали из **26** случаев по ревью и **17** по дополнительному просмотру. Используем этот фактически утверждённый список. [Исходный ноутбук 03b](notebooks/crowd/03b_create_validation.ipynb)

### Обучение ревьюеров

In [ ]:
if MODE == "historical":
    training_tasks = json.loads((inputs / "review/tasks/validation_training.json").read_text())
    assert all(t["input_values"]["photo_id"] in reference_by_id for t in training_tasks)
    print("Учебные задания:", len(training_tasks))
    example_task = training_tasks[0]
    print("Пример:", example_task["input_values"]["photo_id"],
          "| вердикт:", example_task["known_solutions"][0]["output_values"]["verdict"])
else:
    training_id = crowd.open_reviewer_training()
    print("Учебный пул:", training_id)

### Экзамен ревьюеров

In [ ]:
if MODE == "historical":
    exam_pages = json.loads((inputs / "review/tasks/validation_exam.json").read_text())
    review_exam = [task for page in exam_pages for task in page["tasks"]]
    assert all(t["input_values"]["photo_id"] in reference_by_id for t in review_exam)
    answer_key = Counter(t["known_solutions"][0]["output_values"]["verdict"] for t in review_exam)
    print("Экзаменационные задания:", len(review_exam), "| ключ:", dict(answer_key))
else:
    exam_id = crowd.open_reviewer_exam(training_id=training_id)
    print("Экзаменационный пул:", exam_id)

### Независимая проверка первичной разметки

In [ ]:
if MODE == "historical":
    recorded_reviews = indexed(jsonl_rows(root / "data/golden_400/review.jsonl"))
    assert recorded_reviews.keys() == first.keys()
    correction_pages = json.loads((inputs / "annotation/tasks/reannotation.json").read_text())
    correction_tasks = [t["input_values"] for p in correction_pages for t in p["tasks"]]
    correction_ids = {t["photo_id"] for t in correction_tasks}
    assert len(correction_tasks) == len(correction_ids) and correction_ids <= first.keys()
    assert all(main_urls[t["photo_id"]] == t["image"] for t in correction_tasks)
    print("На второй проход по решению команды:", len(correction_ids))
else:
    votes = crowd.review(first, exam_id=exam_id)
    print("Вердикты первого прохода:", dict(Counter(v["majority"] for v in votes.values())))

## 03c. Переразметка спорных фотографий

Повторный ответ привязывается к тому же `photo_id`. Сравним координаты до и после. [Исходный ноутбук 03c](notebooks/crowd/03c_create_reannotation.ipynb)

In [ ]:
if MODE == "historical":
    revised = historical_answers("revalidation.json", correction_ids)
else:
    revised = crowd.reannotate(first, votes)
write_rows(crowd_dir / "second_answers.jsonl", revised.values())
print(f"Получено {len(revised)} повторных ответов, рамки изменились на "
      f"{sum(r['boxes'] != first[i]['boxes'] for i, r in revised.items())} снимках.")

## 03d. Повторная проверка и сборка эталона

В историческом эксперименте заказчик включил все 43 повторных ответа, в том числе 35 с отрицательным большинством. Для точного повтора применяем то же решение и сохраняем голоса отдельно. При новом сборе оставшиеся спорные случаи передаются заказчику на ручное решение. [Исходный ноутбук 03d](notebooks/crowd/03d_create_revalidation.ipynb)

In [ ]:
if MODE == "historical":
    second_votes = {pid: recorded_reviews[pid]["review"] for pid in revised}
else:
    second_votes = crowd.review_corrections(revised)
print("Вердикты после переразметки:", dict(Counter(r["majority"] for r in second_votes.values())))

### Итоговая разметка

In [ ]:
if MODE == "historical":
    final = first | revised
    provenance = []
    for pid, answer in final.items():
        record = recorded_reviews[pid]
        verdict = record["review"]
        assert sum(verdict["votes"].values()) == 3 and verdict["votes"][verdict["majority"]] >= 2
        assert record["status"] == answer["status"]
        assert (record["annotation_source"] == "reannotation_43") == (pid in revised)
        provenance.append(record)
    write_rows(crowd_dir / "review.jsonl", provenance)
    gold_path = crowd_dir / "golden.jsonl"
    write_rows(gold_path, ({k: final[i][k] for k in ("photo_id", "photo_url", "boxes")} for i in sorted(final)))
    restored = indexed(jsonl_rows(gold_path))
    assert restored == indexed(jsonl_rows(root / "data/golden_400/annotations.jsonl"))
    print("Все рамки и URL совпали с историческим эталоном.")
else:
    gold_path = crowd.finish(first, votes, revised, second_votes=second_votes)
gold = indexed(jsonl_rows(gold_path))
assert gold.keys() == first.keys()
print(f"Эталон: фото — {len(gold)}, рамок — {sum(len(r['boxes']) for r in gold.values())}, "
      f"{sum(not r['boxes'] for r in gold.values())} снимков без рамок.")

## 04. Подать одни и те же фотографии четырём моделям

Локальный вход объединяет скачанный оригинал с рамками из предыдущей ячейки по `photo_id`. В историческом режиме восстанавливаем порядок 400 ID и сверяем пиксели с закреплённым файлом Hugging Face, который читал исходный инференс. Его проверочная копия кешируется. Конфигурации фиксируют ревизии весов, промпты, пороги, NMS и повторы. У всех моделей seed **42**. Обучение не требуется. [Исходное демо моделей](notebooks/models/03_model_demo.ipynb) · [Зафиксированные конфигурации](results/final_400/frozen/)

In [ ]:
from scripts.pipeline_models import MODEL_NAMES, prepare_model_manifest, run_model

model_dir = run_dir / "models"
dataset_info = model_archive / "dataset.json"
order = json.loads(dataset_info.read_text())["photo_ids"] if MODE == "historical" else None
manifest = prepare_model_manifest(main_path, gold_path, image_dir, model_dir, photo_ids=order)
if MODE == "historical":
    from scripts.pipeline_models import verify_historical_pixels
    pixel_check = verify_historical_pixels(manifest, dataset_info, Path(os.environ["HF_HOME"]) / "hub")
    print("Проверены входы исходного GPU-прогона:", pixel_check)
print("Вход моделей:", manifest, "| фотографий:", len(jsonl_rows(manifest)))

### Настройки четырёх моделей

In [ ]:
from omegaconf import OmegaConf

settings = []
for name in MODEL_NAMES:
    path = model_archive / "frozen" / f"{name}.yaml"
    expected = [digest for key, digest in frozen_plan["sha256"].items() if key.endswith("/frozen/" + path.name)]
    cfg = OmegaConf.load(path)
    assert expected == [checksum(path)] and cfg.seed == 42
    settings.append([name, cfg.model.backend, cfg.model.revision[:12], cfg.seed,
                     cfg.model.get("threshold", "—"), cfg.postprocess.nms_iou])
table(["Модель", "Бэкенд", "Ревизия весов", "Seed", "Порог", "NMS IoU"], settings)

### Запуск моделей

Как в демонстрационном ноутбуке, выполняем конфигурации последовательно. Здесь каждая модель получает все 400 изображений. В своём каталоге она записывает предсказания, попытки, метрики и журнал.

In [ ]:
results = []
for name in MODEL_NAMES:
    print(f"Запуск {name} на {len(gold)} снимках…", flush=True)
    result = run_model(name, manifest, model_dir)
    results.append(result)
    print(f"{name}: {result['status']}, F1@0.5 = {result['metrics']['f1']:.6f}")

### Пересчитать метрики и выбрать победителя

Сопоставление рамок выполняется отдельно для каждой фотографии при IoU 0,50 и 0,75. Побеждает наибольший object F1 при IoU 0,50. Исторические предсказания читаются только после инференса для сверки. Несовпадение выводится в отчёте: смена GPU, CUDA, версий библиотек или сборки llama.cpp может повлиять на результат даже с тем же seed.

In [ ]:
from butterfly_bench.metrics import evaluate

truth = {i: r["boxes"] for i, r in gold.items()}
summary, predictions = [], {}
for result in results:
    name = result["model"]
    rows = indexed(jsonl_rows(Path(result["output_dir"]) / "predictions.jsonl"))
    assert rows.keys() == truth.keys()
    predictions[name] = rows
    metrics = evaluate(rows, truth)
    assert all(value == result["metrics"][key] for key, value in metrics.items())
    entry = {"model": name, "f1": metrics["f1"], "f1_075": metrics["iou_0.75/f1"],
             "fp": metrics["fp"], "fn": metrics["fn"], "processing_errors": metrics["processing_errors"]}
    if MODE == "historical":
        baseline = indexed(jsonl_rows(model_archive / name / "predictions.jsonl"))
        assert baseline.keys() == rows.keys()
        entry["different_photos"] = sum(row != baseline[i] for i, row in rows.items())
        entry["delta_f1"] = metrics["f1"] - evaluate(baseline, truth)["f1"]
    summary.append(entry)
summary.sort(key=lambda r: -r["f1"])
write_rows(run_dir / "comparison.jsonl", summary)
columns = list(summary[0])
table(columns, [[f"{r[k]:.6f}" if isinstance(r[k], float) else r[k] for k in columns] for r in summary])
winners = [r["model"] for r in summary if r["f1"] == summary[0]["f1"]]
print("Лучший F1@0.5:", ", ".join(winners))

### Посмотреть рамки

In [ ]:
model_inputs = indexed(jsonl_rows(manifest))
example_id = 751308237 if 751308237 in gold else next((i for i, r in gold.items() if r["boxes"]), next(iter(gold)))
with Image.open(manifest.parent / model_inputs[example_id]["image_path"]) as source:
    picture = source.convert("RGB")
fig, axes = plt.subplots(2, 2, figsize=(10, 7), layout="constrained")
for axis, name in zip(axes.flat, MODEL_NAMES):
    annotated = draw_prediction(picture, predictions[name][example_id], truth[example_id])
    annotated.thumbnail((720, 500))
    axis.imshow(annotated)
    axis.set_title(f"{name} · {example_id}")
    axis.axis("off")
plt.show()

### Сырые ответы и ошибки

In [ ]:
winning_run = next(r for r in results if r["model"] == winners[0])
attempts = jsonl_rows(Path(winning_run["output_dir"]) / "attempts.jsonl")
examples = [a for a in attempts if a["photo_id"] == example_id]
table(["Фото", "Попытка", "Seed", "Этап", "Ошибка"],
      [[a["photo_id"], a.get("attempt"), a.get("seed"), a.get("stage"), a.get("error") or "—"]
       for a in examples])
if examples:
    last = examples[-1]
    print(str(last.get("raw_text") or last.get("raw_record") or last.get("error"))[:1200])
print("Сравнение четырёх моделей:", run_dir / "comparison.jsonl")